# 15 — Decision-consequence audit

**Objective.** Translate outcome/explanation changes into top-K overlap, rank reversals, utility, workload, selection composition, and anonymized high-confidence construct-fragile cases.

**Scientific contract.** This notebook reports no empirical result until it executes successfully against hash-verified inputs. It writes immutable outputs plus a completion manifest. Expected counts are protocol assertions, not substituted observations.

Geography and industry are selection-distribution audits, not demographic fairness proxies. No named startup is released.

In [ ]:
# Standard CRUX-VC Colab bootstrap. Git dotfiles restore from the Drive project root; tokens never appear in cells.
import os, subprocess, sys
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass

import shutil
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/CRUX_Research")
for _dotfile in (".gitconfig", ".git-credentials"):
    if (DRIVE_PROJECT_ROOT / _dotfile).exists():
        shutil.copy(DRIVE_PROJECT_ROOT / _dotfile, Path.home() / _dotfile)
if (Path.home() / ".git-credentials").exists():
    os.chmod(Path.home() / ".git-credentials", 0o600)

REPO_URL = "https://github.com/anasbiswas1/crux-vc"
REPO_ROOT = Path(os.environ.get("CRUX_REPO_ROOT", "/content/drive/MyDrive/CRUX_Research/crux-vc"))
if not (REPO_ROOT / ".cruxvc-root").exists():
    if REPO_ROOT.exists() and any(REPO_ROOT.iterdir()):
        raise RuntimeError(f"{REPO_ROOT} exists but is not a CRUX-VC checkout")
    REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

try:
    import yaml, pandas, sklearn, pyarrow  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")], check=True)

from cruxvc.runtime import bootstrap_notebook
CTX = bootstrap_notebook("15", suffix=None)
P, CFG, PROFILE = CTX.paths, CTX.config, CTX.profile

In [ ]:
import os
import pandas as pd
from cruxvc.decision import expected_utility, portfolio_overlap_audit, selection_performance, subgroup_selection_composition
from cruxvc.io import read_table, write_table
from cruxvc.manifest import append_test_access_log, signing_key_from_environment
from cruxvc.validation import anonymize_case_ids

append_test_access_log(P, stage_id="15", purpose="locked decision-consequence audit", resources=[P.predictions / "final_predictions.parquet"])
predictions = read_table(P.predictions / "final_predictions.parquet")
cohort = read_table(P.processed / "cohort_labels.parquet")
extended = read_table(P.processed / "features_extended.parquet")
calibration_registry = read_table(P.models / "calibration_registry.parquet")
reference = calibration_registry[calibration_registry["analysis_role"].eq("matched_reference")].copy()
primary_family = reference[reference["outcome"].eq("F36")].sort_values("platt_oof_log_loss").iloc[0]["family"]
matched = predictions[
    predictions["analysis_role"].eq("matched_reference_deployment")
    & predictions["family"].eq(primary_family)
    & predictions["outcome"].isin(CFG["outcomes"]["confirmatory"])
].copy()


In [ ]:
budgets = [*CFG["metrics"]["review_budgets"]]
if float(CFG["metrics"]["stakeholder_budget"]) not in budgets:
    budgets.append(float(CFG["metrics"]["stakeholder_budget"]))
overlap = portfolio_overlap_audit(matched, budgets=budgets)
performance = selection_performance(matched, budgets=budgets)
utility = expected_utility(
    matched, budgets=budgets,
    false_positive_costs=[0.1, 0.25, 0.5, 1.0],
    false_negative_costs=[0.5, 1.0, 2.0, 5.0],
)
metadata = extended[["case_id", "category_code", "country_code", "region", "landmark_round_type"]]
composition = subgroup_selection_composition(
    matched, metadata, subgroup_columns=["category_code", "country_code", "region", "landmark_round_type"],
    budgets=budgets, minimum_cell=10,
)

In [ ]:
rq1_cross = read_table(P.inference / "rq1_cross_spec_distances.parquet")
fragility = rq1_cross.groupby(["case_id", "contrast"])["sqrt_jsd"].mean().unstack().mean(axis=1).rename("construct_fragility")
confidence = matched.groupby("case_id")["probability"].max().rename("maximum_screening_probability")
cases = pd.concat([fragility, confidence], axis=1).dropna().reset_index()
cases = cases.sort_values(["maximum_screening_probability", "construct_fragility"], ascending=False).head(20)
release_salt = os.environ.get("CRUX_RELEASE_SALT") or signing_key_from_environment()
if not release_salt:
    raise RuntimeError(
        "Case-study anonymization requires CRUX_RELEASE_SALT or CRUX_PROTOCOL_SIGNING_KEY. "
        "Store it in Colab Secrets; never commit it to the repository."
    )
cases = anonymize_case_ids(cases, salt=release_salt)


In [ ]:
overlap_path = write_table(overlap, P.decision / "topk_portfolio_overlap.csv")
performance_path = write_table(performance, P.decision / "screening_performance.csv")
utility_path = write_table(utility, P.decision / "expected_utility_grid.csv")
composition_path = write_table(composition, P.decision / "selection_composition.csv")
case_path = write_table(cases, P.decision / "anonymized_construct_fragile_cases.csv")
CTX.recorder.complete([overlap_path, performance_path, utility_path, composition_path, case_path])
print(overlap.to_string(index=False))